In [1]:
import pandas as pd
import numpy as np
from VaRData import VarData, CONFIGS
from VaRMethods import GarchSim
from VaRBacktesting import VarBacktest
from datetime import datetime
import scipy.stats
import warnings

### Functions & Constants

In [2]:
def get_lists():
    """creates lists of volatility_factor_ids and tickers for iterations
    
    Reurns: dict
    
    """
    volatility_factor_id, tickers, weights, ndays = [], [], [], []

    for idx, vals in CONFIGS["var_settings"].items():
        volatility_factor_id.append(idx)
        tickers.append(vals["tickers"])
        weights.append(None if vals["weights"] == "None" else vals["weights"])
        ndays.append(vals["ndays"])

    return {
        "ids": volatility_factor_id, 
        "tickers": tickers,
        "weights": weights,
        "ndays": ndays
        }

def F_monitor(index, period_full, period_start, period_new, alpha=0.99):
    """calculates F-Stats and critical value
    
    Args:
        period_full (list): list of start / end date of full sample period
        period_1yr (list): list of start / end date of last year in sample period
        period_new (list): list of start / end date of new period

    Returns: dict (see key:values below)
        Q-vs-Full: variance of out-of-sample period vs. variance of full sample
        Q-vs-1Y: variance of out-of-sample period vs. variance of last year in sample 
        Q-vs-Max: variance of out-of-sample period vs. largest n-day variance of full sample period;
            where n equals the number of days in the period_new
    """
    df = VarData(index).wtd_log_return

    full = df.loc[period_full[0]:period_full[1]]    #full sample
    start = df.loc[period_start[0]:period_start[1]] #last year (starting values for simulation)
    new  = df.loc[period_new[0]:period_new[1]]      #new out-of-sample period to calculate f-stats

    return {
        "Q-vs-Full":     new.values.var(ddof = 1) / full.values.var(ddof = 1),                                    
        "Q-vs-Full_CV":  scipy.stats.f.ppf(q = alpha, dfn = len(new) - 1, dfd = len(full) - 1),                  
        "Q-vs-1Y":    new.values.var(ddof = 1) / start.values.var(ddof = 1),                                  
        "Q-vs-1Y_CV": scipy.stats.f.ppf(q = alpha, dfn = len(new) - 1, dfd = len(start) - 1),                   
        "Q-vs-Max":      new.values.var(ddof = 1) / full.rolling(len(new)).var().dropna().quantile(1).values[0],  
        "Q-vs-Max_CV":   scipy.stats.f.ppf(q = alpha, dfn = len(new)-1, dfd = len(new) - 1)                   
    }

def LogLikelihood_test(break_ser, converge_rate = 0.01, alpha=0.99):

    T = len(break_ser)
    N = break_ser.sum()

    # Kupiec(1995)
    if N == T:
        LR_uc = float("inf")
    else:
        LR_uc = -2 * np.log((1-converge_rate)**(T-N)*(converge_rate**N)) + 2 * np.log((1-N/T)**(T-N)*(N/T)**N)

    # Christoffersen (1998)
    df_break = break_ser
    T00,T01,T10,T11 = 0,0,0,0
    for i in range(len(df_break)-1):
        if i == 0:
            if df_break[0] == 0:
                T00 = T00 + 1
            else:
                T01 = T01 + 1
        if df_break[i] == 0:
            if df_break[i+1] == 0:
                T00 = T00 + 1
            else:
                T01 = T01 + 1
        elif df_break[i] == 1:
            if df_break[i+1] == 0:
                T10 = T10 + 1
            else:
                T11 = T11 + 1   
    
    if N == T:
        LR_ind = float("inf")
    elif N == 0:
        LR_ind = 0
    else:
        if T00 == 0:
            pi0 = 1
        else:
            pi0 = T01 / (T - N)
        pi  = N / T
        pi1 = T11 / N
        LR_ind = -2 * np.log((1-pi)**(T00+T10) * pi**(T01+T11)) + 2 * np.log((1-pi0)**T00 * pi0**T01 * (1-pi1)**T10 * pi1**T11)
  
    LR_cc = LR_uc + LR_ind
    
    chi2_1 = scipy.stats.chi2.ppf(alpha,1)
    chi2_2 = scipy.stats.chi2.ppf(alpha,2)

    result = {
        'Exceptions': break_ser.sum(),
        'LR_uc':round(LR_uc,3), 
        'LR_ind':round(LR_ind,3), 
        'LR_cc':round(LR_cc,3), 
        f'chi2(1) {int(alpha*100)}%':round(chi2_1,3), 
        f'chi2(2) {int(alpha*100)}%':round(chi2_2,3),
        "Independence": "RejectNull" if LR_ind > chi2_1 else None,
        "alpha": round(alpha * 100, 1),
        "converge_rate": round(converge_rate * 100,1)}

    return result

#### 1. Set Parameters

In [3]:
PERIOD_FULL   = ['2016-01-01', '2024-12-31'] #full sample period
PERIOD_START  = ['2024-01-01', '2024-12-31'] #last year of sample period
PERIOD_NEW_1  = ['2025-01-01', '2025-03-31'] #Q1
PERIOD_NEW_2  = ['2025-04-01', '2025-06-30'] #Q2
PERIOD_NEW_3  = ['2025-07-01', '2025-09-30'] #Q3
PERIOD_NEW_4  = ['2025-10-01', '2025-12-31'] #Q4
PERIOD_NEW_5  = ['2026-01-01', '2026-03-31'] #Q5

#### 2. F-Test

##### 2.1 Generate F-Statistics and Critical Values

In [4]:
output_name = "FStats_20250930.xlsx"
save_output = True
quarters = [1,2,3]

_lists = get_lists()
tickers, volatility_factor_id = _lists.get("tickers"), _lists.get("ids")

frames = [] #quarter-level results
for qtr in quarters:
    results = [] #ticker-level results
    for idx in range(len(tickers)): 
        temp = {"ID": volatility_factor_id[idx]}
        period_new = None #set initial value for period_new
        if qtr == 1:
            period_new = PERIOD_NEW_1
        elif qtr == 2:
            period_new = PERIOD_NEW_2
        elif qtr == 3:
            period_new = PERIOD_NEW_3
        elif qtr == 4:
            period_new = PERIOD_NEW_4 
        elif qtr == 5:
            period_new = PERIOD_NEW_5 
        results.append({**temp, **F_monitor(index = tickers[idx], period_full = PERIOD_FULL, period_start = PERIOD_START, period_new = period_new)}) 

    df = pd.DataFrame.from_dict(results)
    df.columns = [str(qtr) + col if col != "ID" else col for col in df.columns] 
    frames.append(df.set_index("ID"))
    print(f"---------------Results for Q{qtr}-----------------")
    display(df.head())

#---------------Comparison---------------#
comp = pd.concat(frames, axis=1)
print(f"---------------Results for ALL Quarters in qtr-----------------")
display(comp.head())

if save_output == True:
    comp.to_excel("output//" + output_name, index=True)

---------------Results for Q1-----------------


,ID,1Q-vs-Full,1Q-vs-Full_CV,1Q-vs-1Y,1Q-vs-1Y_CV,1Q-vs-Max,1Q-vs-Max_CV
0,agy_bond_fixed_1to3,0.547244,1.482822,0.552735,1.561758,0.074424,1.836259
1,agy_bond_fixed_3to5,2.478707,1.482822,2.142853,1.561758,0.453298,1.836259
2,agy_bond_fixed_gt5,0.905228,1.482822,0.953202,1.561758,0.213853,1.836259
3,agy_cmo,0.962261,1.482822,0.821909,1.561758,0.210018,1.836259
4,agy_cmo_excl_strip,1.092522,1.482822,0.805225,1.561758,0.256570,1.836259


---------------Results for Q2-----------------


,ID,2Q-vs-Full,2Q-vs-Full_CV,2Q-vs-1Y,2Q-vs-1Y_CV,2Q-vs-Max,2Q-vs-Max_CV
0,agy_bond_fixed_1to3,0.899520,1.478582,0.908545,1.557785,0.124157,1.826886
1,agy_bond_fixed_3to5,2.081968,1.478582,1.799870,1.557785,0.385755,1.826886
2,agy_bond_fixed_gt5,1.291875,1.478582,1.360339,1.557785,0.309873,1.826886
3,agy_cmo,1.200917,1.478582,1.025755,1.557785,0.265531,1.826886
4,agy_cmo_excl_strip,1.372828,1.478582,1.011819,1.557785,0.326839,1.826886


---------------Results for Q3-----------------


,ID,3Q-vs-Full,3Q-vs-Full_CV,3Q-vs-1Y,3Q-vs-1Y_CV,3Q-vs-Max,3Q-vs-Max_CV
0,agy_bond_fixed_1to3,0.477510,1.470425,0.482301,1.550158,0.067100,1.808962
1,agy_bond_fixed_3to5,0.885830,1.470425,0.765804,1.550158,0.167202,1.808962
2,agy_bond_fixed_gt5,0.580613,1.470425,0.611383,1.550158,0.143250,1.808962
3,agy_cmo,0.680347,1.470425,0.581114,1.550158,0.154334,1.808962
4,agy_cmo_excl_strip,0.769326,1.470425,0.567019,1.550158,0.187222,1.808962


---------------Results for ALL Quarters in qtr-----------------


,1Q-vs-Full,1Q-vs-Full_CV,1Q-vs-1Y,1Q-vs-1Y_CV,1Q-vs-Max,1Q-vs-Max_CV,2Q-vs-Full,2Q-vs-Full_CV,2Q-vs-1Y,2Q-vs-1Y_CV,2Q-vs-Max,2Q-vs-Max_CV,3Q-vs-Full,3Q-vs-Full_CV,3Q-vs-1Y,3Q-vs-1Y_CV,3Q-vs-Max,3Q-vs-Max_CV
ID,,,,,,,,,,,,,,,,,,
agy_bond_fixed_1to3,0.547244,1.482822,0.552735,1.561758,0.074424,1.836259,0.899520,1.478582,0.908545,1.557785,0.124157,1.826886,0.477510,1.470425,0.482301,1.550158,0.067100,1.808962
agy_bond_fixed_3to5,2.478707,1.482822,2.142853,1.561758,0.453298,1.836259,2.081968,1.478582,1.799870,1.557785,0.385755,1.826886,0.885830,1.470425,0.765804,1.550158,0.167202,1.808962
agy_bond_fixed_gt5,0.905228,1.482822,0.953202,1.561758,0.213853,1.836259,1.291875,1.478582,1.360339,1.557785,0.309873,1.826886,0.580613,1.470425,0.611383,1.550158,0.143250,1.808962
agy_cmo,0.962261,1.482822,0.821909,1.561758,0.210018,1.836259,1.200917,1.478582,1.025755,1.557785,0.265531,1.826886,0.680347,1.470425,0.581114,1.550158,0.154334,1.808962
agy_cmo_excl_strip,1.092522,1.482822,0.805225,1.561758,0.256570,1.836259,1.372828,1.478582,1.011819,1.557785,0.326839,1.826886,0.769326,1.470425,0.567019,1.550158,0.187222,1.808962


#### 3. Exceptions

##### 3.1 Generate 1-Day Exception Test Results

Code-block can be executed for any of the performance monitoring numbers.  The results (1-day exceptions) are delineated by the quarter in which they occur.

In [5]:
output_name = "OneDayExceptions_20250630.xlsx"
save_output = True

_lists = get_lists()
tickers, volatility_factor_id, weights = _lists.get("tickers"), _lists.get("ids"), _lists.get("weights")

r1, r2, r3, r4 = [], [], [], []
nday_vars = []
for i in range(len(tickers)):
    v = GarchSim(tickers = tickers[i], weights = weights[i], sample_start = PERIOD_FULL[0], sample_end = PERIOD_FULL[1]) #instantiate GarchSim
    var = v.CustomNDayPriceSimulation(horizon1 = 63, horizon2 = 252, ndays = 1, nsims = 10000, 
                                    shock_type = "custom_normal", start_vals=[PERIOD_START[0], 0.50])
    nday_var = var["VaR_95"]
    bt1 = VarBacktest(tickers = tickers[i], n_liquidation = 1, VaR = nday_var, start_date=PERIOD_NEW_1[0], end_date=PERIOD_NEW_1[1], VaR_is_log = False, forward_shift = False) #Q1
    bt2 = VarBacktest(tickers = tickers[i], n_liquidation = 1, VaR = nday_var, start_date=PERIOD_NEW_2[0], end_date=PERIOD_NEW_2[1], VaR_is_log = False, forward_shift = False) #Q2
    bt3 = VarBacktest(tickers = tickers[i], n_liquidation = 1, VaR = nday_var, start_date=PERIOD_NEW_3[0], end_date=PERIOD_NEW_3[1], VaR_is_log = False, forward_shift = False) #Q3
    bt4 = VarBacktest(tickers = tickers[i], n_liquidation = 1, VaR = nday_var, start_date=PERIOD_NEW_4[0], end_date=PERIOD_NEW_4[1], VaR_is_log = False, forward_shift = False) #Q4

    r1.append(bt1.break_series.sum())
    r2.append(bt2.break_series.sum())
    r3.append(bt3.break_series.sum())
    r4.append(bt4.break_series.sum())

    nday_vars.append(nday_var)

df = pd.DataFrame({#
    "ID":volatility_factor_id, 
    "1-Day_VaR": nday_vars,
    "Exception_Q1": r1, 
    "Exception_Q2": r2, 
    "Exception_Q3": r3, 
    "Exception_Q4": r4, 
    "Total_Exceptions": np.array(r1) + np.array(r2) + np.array(r3) + np.array(r4)
    })

display(df)
if save_output == True:
    df.to_excel("output//" + output_name, index=False)

,ID,1-Day_VaR,Exception_Q1,Exception_Q2,Exception_Q3,Exception_Q4,Total_Exceptions
0,agy_bond_fixed_1to3,0.2800,0,0,0,0,0
1,agy_bond_fixed_3to5,0.6147,0,0,0,0,0
2,agy_bond_fixed_gt5,1.5273,0,0,0,0,0
3,agy_cmo,0.9441,0,1,0,0,1
4,agy_cmo_excl_strip,1.9070,0,0,0,0,0
5,agy_cmo_z,2.3360,0,0,0,0,0
6,cmbs_fixed_aa,0.7701,0,0,0,0,0
7,cmbs_fixed_aaa,0.9234,0,0,0,0,0
8,muni_go_aa,2.2753,0,0,0,0,0
9,muni_go_aaa,1.1440,0,4,0,0,4


##### 3.2 Generate N-Day Exception Test Results

Code-block can be executed for only exceptions reporting for the 3rd, 4th, and 5th performance monitoring reports.  The results (n-day exceptions) are those observed over the period defined by the variable 'quarter'.

In [5]:
quarter = 3 #set to one of: 3, 4, 5 
output_name = "Exceptions_20250930.xlsx"
save_output = True

ExceptionAdjustment = {
    0:0,1:0, 2:1, 3:2, 4:2, 5:2, 6:2, 7:3, 8:3, 9:3, 10:3, 11:4, 12:4, 13:4, 14:5, 15:5, 16:5,
    17:5, 18:6, 19:6, 20:6, 21:7, 22:7, 23:8, 24:8, 25:8, 26:9, 27:9, 28:1, 29:1, 30:1, 31:11,
    32:11, 33:12, 34:12, 35:13, 36:36, 37:14, 38:14, 39:15, 40:15
    }
    
_lists = get_lists()
tickers, volatility_factor_id, weights, ndays = _lists.get("tickers"), _lists.get("ids"), _lists.get("weights"), _lists.get("ndays")

frames = []
for i in range(len(tickers)):
    for y in range(len(ndays[i])):

        #--------------Step 1: Calculate VaR--------------------#
        v = GarchSim(tickers = tickers[i], weights = weights[i], sample_start = PERIOD_FULL[0], sample_end = PERIOD_FULL[1]) #instantiate GarchSim
        var = v.CustomNDayPriceSimulation(horizon1 = 63, horizon2 = 252 + ndays[i][y] - 1, ndays = ndays[i][y], nsims = 10000, 
                                        shock_type = "custom_normal", start_vals=[PERIOD_START[0], 0.50])
        nday_var = var["VaR_95"]

        #------------Step 2: Generate backtesting results-------#
        end_date = None #Set initial value for end-date
        if quarter == 3:
            end_date = PERIOD_NEW_3[1]
        elif quarter == 4:
            end_date = PERIOD_NEW_4[1]
        elif quarter == 5:
            end_date = PERIOD_NEW_5[1]      

        bt = VarBacktest(tickers = tickers[i], n_liquidation = ndays[i][y], VaR = nday_var, start_date=PERIOD_NEW_2[0], end_date=end_date, VaR_is_log = False, forward_shift = False)
        ind = LogLikelihood_test(bt.break_series)

        TotalExceptions = bt.break_series.sum()
        IsDependent = True if ind.get("Independence") == "RejectNull" else False
        if IsDependent == False:
            Adjustment = 0
        elif TotalExceptions > 40:
            Adjustment = 15
        else:
            Adjustment = ExceptionAdjustment.get(TotalExceptions)

        #Step 3: Create backtesting summary dataframe
        frames.append(pd.DataFrame({
            "Volatility_Factor_ID": volatility_factor_id[i],
            "Tickers": ", ".join(tickers[i]) if len(tickers[i]) > 1 else tickers[i],
            "Weights": ", ".join([str(x) for x in weights[i]]) if weights[i] != None else '1',
            "NDays": ndays[i][y],
            "NDayVaR": nday_var,
            "ForecastStart": bt.start_date,
            "ForecastEnd": bt.end_date,
            "TotalExceptions": TotalExceptions,
            "IndependenceTest": ind.get("Independence"),
            "IndependenceAdjustment": Adjustment,
            "TotalExceptionsAdjusted": TotalExceptions - Adjustment
        }, index = [0]))

df = pd.concat(frames)

display(df)
if save_output == True:
    df.to_excel("output//" + output_name, index=False)

,Volatility_Factor_ID,Tickers,Weights,NDays,NDayVaR,ForecastStart,ForecastEnd,TotalExceptions,IndependenceTest,IndependenceAdjustment,TotalExceptionsAdjusted
0,agy_bond_fixed_1to3,G1P0,1,5,0.6447,2025-04-01,2025-09-30,0,None,0,0
0,agy_bond_fixed_1to3,G1P0,1,10,0.9477,2025-04-01,2025-09-30,0,None,0,0
0,agy_bond_fixed_1to3,G1P0,1,252,3.3538,2025-04-01,2025-09-30,0,None,0,0
0,agy_bond_fixed_3to5,G2P0,1,5,1.4310,2025-04-01,2025-09-30,0,None,0,0
0,agy_bond_fixed_3to5,G2P0,1,10,2.0694,2025-04-01,2025-09-30,0,None,0,0
...,...,...,...,...,...,...,...,...,...,...,...
0,prime_arm_mbs,"PRIMEX.ARM.1, PRIMEX.ARM.2",1,252,10.4884,2025-04-01,2025-09-30,0,None,0,0
0,prime_mbs,PRIME.13H1,1,5,3.7783,2025-04-01,2025-09-30,0,None,0,0
0,prime_mbs,PRIME.13H1,1,10,5.3018,2025-04-01,2025-09-30,0,None,0,0
0,prime_mbs,PRIME.13H1,1,84,10.5220,2025-04-01,2025-09-30,0,None,0,0


##### 3.3 Generate Shortfall Test Results

Code-block can be executed for only performance monitoring in Q3 and Q4.  The results (shortalls for the given index) are those observed over the period defined by the variable 'quarter'.

In [9]:
quarter = 3 #set to one of: 3, 4
ticker = "G0BA"
weights = None
ndays = 10
output_name = f"{ticker}_{weights}_{ndays}_shortfall_20230930.xlsx"
save_output = True 

v = GarchSim(tickers = ticker, weights = weights, sample_start = PERIOD_FULL[0], sample_end = PERIOD_FULL[1]) #instantiate GarchSim
var = v.CustomNDayPriceSimulation(horizon1 = 63, horizon2 = 252 + ndays - 1, ndays = ndays, nsims = 10000, 
                                shock_type = "custom_normal", start_vals=["2022-01-01", 0.50])
end_date = None #Set initial value for end-date
if quarter == 3:
    end_date = PERIOD_NEW_3[1]
elif quarter == 4:
    end_date = PERIOD_NEW_4[1]
bt = VarBacktest(tickers = ticker, n_liquidation = ndays, VaR = var["VaR_95"], start_date=PERIOD_NEW_2[0], end_date=end_date, VaR_is_log = False, forward_shift = False)

df = bt.shortfall_series.to_frame(name = ticker)

display(df)

if save_output == True:
    df.to_excel("output//" + output_name, index=True)

,G0BA
Date,
2023-04-03,0.000618
2023-04-04,0.000719
2023-04-05,0.000821
2023-04-06,0.000619
2023-04-10,0.000519
...,...
2023-09-25,0.004968
2023-09-26,0.004967
2023-09-27,0.004967
